# Oversampled replay of the three 21 August submissions

This notebook compares the exact three submitted recipes under their original training distribution, the pre-declared full oversampling policy, an 80%-replica follow-up and one final 2.5× minority-count point. The labelled **local test** supplies the recorded accuracy; the unlabelled DrivenData competition set supplies only prediction shares. No new submissions are uploaded here.

## Course-aligned ten-step lifecycle

| Step | Treatment |
|---:|---|
| 1. Define the goal and scope | Test whether minority replication is credible for a later submission, using ordinary accuracy and class-aware diagnostics. |
| 2. Gather the data | Reuse the immutable labelled, competition and submission-template files plus the three generated 21 August CSVs. |
| 3. Explore the data | Compare natural, fitted and predicted class distributions. |
| 4. Clean and preprocess the data | Reuse each frozen recipe's fold-fitted preprocessing; change no predictors. |
| 5. Select and engineer features | Not applicable: feature policy is frozen to isolate the resampling intervention. |
| 6. Define the machine-learning task | Three-class nominal classification; oversampling changes the fit distribution, not the target contract. |
| 7. Partition the data | Fit on frozen development rows and score the labelled local test; refit on all labelled rows for competition predictions. |
| 8. Select and train candidate methods | Replay the exact Random Forest/histogram and XGBoost/Random Forest recipes with fixed tree counts. |
| 9. Evaluate and interpret the results | Compare accuracy, balanced accuracy, macro-F1, class recall and prediction shares. |
| 10. Deploy and iterate | Write validated candidate CSVs; upload is deliberately deferred. |

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

stage_directory = next(
    candidate for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'data' / 'TrainingSetValues.csv').is_file()
)
project_directory = stage_directory.parent
runtime_directory = project_directory / '.runtime'
result_paths = {
    '100% of added replicas': (
        runtime_directory / 'oversampled-submission-replay' / 'results.json'
    ),
    '80% of added replicas': (
        runtime_directory / 'oversampled-submission-replay-80' / 'results.json'
    ),
    '2.5x final repair count': (
        runtime_directory / 'oversampled-submission-replay-250x' / 'results.json'
    ),
}
results = {
    name: json.loads(path.read_text(encoding='utf-8'))
    for name, path in result_paths.items()
}
recipe_labels = {
    'random-forest-histogram-boosting': '50% RF + 50% histogram',
    '55-xgboost-depth-8-child-1-current-one-hot-45-random-forest': (
        '55% XGB depth 8 child 1 + 45% RF'
    ),
    '60-xgboost-depth-6-7-8-local-bag-40-random-forest': (
        '60% XGB depth bag + 40% RF'
    ),
}
print('Loaded both verified replay records.')

Loaded both verified replay records.


## How much oversampling was applied?

Only `functional needs repair` is replicated, with replacement and a fixed seed. The 100% policy raises it to the natural count of the second-largest class, `non functional`. The 80% policy retains 80% of the **added replicas**, rather than multiplying the final minority count by 0.8. Validation, local-test and competition rows retain their natural distributions.

In [2]:
extent_rows = []
for regime, payload in results.items():
    detail = payload['oversampling']
    for scope in ('development', 'full_labelled'):
        before = detail[f'{scope}_counts_before']
        after = detail[f'{scope}_counts_after']
        rows_after = detail[f'{scope}_rows_after']
        extent_rows.append(
            {
                'regime': regime,
                'scope': scope.replace('_', ' '),
                'rows before': detail[f'{scope}_rows_before'],
                'rows after': rows_after,
                'repair before': before['functional needs repair'],
                'repair after': after['functional needs repair'],
                'repair share after': (
                    after['functional needs repair'] / rows_after
                ),
            }
        )
oversampling_extent = pd.DataFrame(extent_rows).set_index(['regime', 'scope'])
display(oversampling_extent.style.format({'repair share after': '{:.2%}'}))

## Labelled local-test comparison

The 100% policy was fixed before this replay. The 80% point and final 2.5× point were proposed after seeing earlier replay results; their local-test figures are therefore **adaptive exploratory evidence**, not independent tests. The sequence stops at 2.5×: no neighbouring multiplier or calibration adjustment is searched against this local test.

In [3]:
local_rows = []
original_payload = results['100% of added replicas']
for recipe, label in recipe_labels.items():
    original = original_payload['local_test'][recipe]['original_training']
    local_rows.append(
        {
            'recipe': label,
            'training': 'original',
            'accuracy': original['accuracy'],
            'balanced accuracy': original['balanced_accuracy'],
            'macro F1': original['macro_f1'],
            'repair recall': original['class_recall']['functional needs repair'],
            'predicted repair share': (
                original['prediction_share']['functional needs repair']
            ),
        }
    )
    for regime, payload in results.items():
        measured = payload['local_test'][recipe]['oversampled_training']
        local_rows.append(
            {
                'recipe': label,
                'training': regime,
                'accuracy': measured['accuracy'],
                'balanced accuracy': measured['balanced_accuracy'],
                'macro F1': measured['macro_f1'],
                'repair recall': (
                    measured['class_recall']['functional needs repair']
                ),
                'predicted repair share': (
                    measured['prediction_share']['functional needs repair']
                ),
            }
        )
local_comparison = pd.DataFrame(local_rows).set_index(['recipe', 'training'])
display(local_comparison.style.format('{:.2%}'))

## Competition prediction distributions

The competition labels are hidden, so these rows do **not** provide competition accuracy. They show how the intervention changes the submitted label mix relative to today's exact CSVs.

In [4]:
competition_rows = []
for recipe, label in recipe_labels.items():
    original = original_payload['competition_predictions'][recipe]['original']
    competition_rows.append(
        {
            'recipe': label,
            'training': 'original',
            **original['prediction_share'],
        }
    )
    for regime, payload in results.items():
        replay = payload['competition_predictions'][recipe]['oversampled']
        competition_rows.append(
            {
                'recipe': label,
                'training': regime,
                **replay['prediction_share'],
            }
        )
competition_shares = pd.DataFrame(competition_rows).set_index(
    ['recipe', 'training']
)
display(competition_shares.style.format('{:.2%}'))

## Decision for a possible later submission

Oversampling is now a credible **minority-recall intervention**, not an evidence-backed accuracy improvement. Across the exact recipes, the 100% policy loses 1.4–1.8 percentage points of local-test accuracy but gains 20.0–22.4 points of repair recall. Retaining 80% of the replicas recovers a little accuracy. The final 2.5× point gives the cleanest trade-off: its XGBoost blends lose only 0.40–0.43 accuracy points, gain 10.9–11.9 repair-recall points, and produce competition repair shares of 6.98–7.10%, close to the independently inferred 7.19% prevalence.

If one later upload slot is explicitly reserved for a distribution-shift experiment, the 2.5× child-weight-1 XGBoost/Random Forest variant is the cleanest candidate. It should not replace the stronger original recipe on the claim that it is more accurate: only a DrivenData score can test that, and the labelled local-test evidence remains slightly lower. Further multipliers or probability calibration must be selected inside development cross-validation rather than fitted to this already-observed local test.

In [5]:
assert all(path.is_file() for path in result_paths.values())
assert len(local_comparison) == 12
assert len(competition_shares) == 12
assert oversampling_extent.loc[
    ('80% of added replicas', 'full labelled'), 'repair after'
] == 19_123
assert oversampling_extent.loc[
    ('2.5x final repair count', 'full labelled'), 'repair after'
] == 10_793
assert competition_shares.index.is_unique
print('All replay records, row counts and comparison tables verified.')

All replay records, row counts and comparison tables verified.
